In [1]:
# imports
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

In [2]:
# loading the model
class MLPExperiment2(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(3, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 2),
        )

    def forward(self, x):

        return self.network(x)

In [3]:
model = MLPExperiment2()

model.load_state_dict(
    torch.load("mlp_exp2_model.pth", map_location=torch.device("cpu"))
)

model.eval()

print("MLP Loaded")

MLP Loaded


In [4]:
# loading the scaler
scaler = joblib.load("mlp_scaler_exp2.pkl")

print("Scaler Loaded")

Scaler Loaded


In [5]:
# loading datasets
datasets = {
    "Degraded": "degraded_mlp.csv",
    "Clean": "clean_mlp.csv",
    "Natural": "natural_mlp.csv",
    "Unknown": "unknown_mlp.csv",
}

In [6]:
# evaluation function
def evaluate_dataset(csv_file):

    df = pd.read_csv(csv_file)

    X = df[["quality_score", "best_similarity", "margin"]]

    y_true = df["label"]

    X_scaled = scaler.transform(X)

    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

    with torch.no_grad():

        outputs = model(X_tensor)

        y_pred = torch.argmax(outputs, dim=1).numpy()

    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(y_true, y_pred, zero_division=0)

    recall = recall_score(y_true, y_pred, zero_division=0)

    f1 = f1_score(y_true, y_pred, zero_division=0)

    cm = confusion_matrix(y_true, y_pred)

    if cm.shape == (2, 2):

        TN, FP, FN, TP = cm.ravel()

        TAR = TP / (TP + FN) if (TP + FN) > 0 else np.nan

        FRR = FN / (TP + FN) if (TP + FN) > 0 else np.nan

        TRR = TN / (TN + FP) if (TN + FP) > 0 else np.nan

        FAR = FP / (TN + FP) if (TN + FP) > 0 else np.nan

    else:

        TAR = np.nan
        FRR = np.nan
        TRR = np.nan
        FAR = np.nan

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "TAR": TAR,
        "FRR": FRR,
        "TRR": TRR,
        "FAR": FAR,
    }

In [7]:
# threshold evaluation
def evaluate_threshold(csv_file):

    df = pd.read_csv(csv_file)

    y_true = df["label"]

    threshold_pred = (df["best_similarity"] >= 0.6).astype(int)

    accuracy = accuracy_score(y_true, threshold_pred)

    precision = precision_score(y_true, threshold_pred, zero_division=0)

    recall = recall_score(y_true, threshold_pred, zero_division=0)

    f1 = f1_score(y_true, threshold_pred, zero_division=0)

    cm = confusion_matrix(y_true, threshold_pred)

    if cm.shape == (2, 2):

        TN, FP, FN, TP = cm.ravel()

        TAR = TP / (TP + FN) if (TP + FN) > 0 else np.nan

        FRR = FN / (TP + FN) if (TP + FN) > 0 else np.nan

        TRR = TN / (TN + FP) if (TN + FP) > 0 else np.nan

        FAR = FP / (TN + FP) if (TN + FP) > 0 else np.nan

    else:

        TAR = np.nan
        FRR = np.nan
        TRR = np.nan
        FAR = np.nan

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "TAR": TAR,
        "FRR": FRR,
        "TRR": TRR,
        "FAR": FAR,
    }

In [8]:
# running our multi layer MLP
mlp_exp2_results = {}

for dataset_name, csv_file in datasets.items():

    result = evaluate_dataset(csv_file)

    mlp_exp2_results[dataset_name] = result

    print("\n======================")
    print(dataset_name)
    print("======================")

    print(pd.DataFrame([result]).round(4))


Degraded
   Accuracy  Precision  Recall      F1     TAR     FRR     TRR     FAR
0    0.9685     0.9981  0.9697  0.9837  0.9697  0.0303  0.9167  0.0833

Clean
   Accuracy  Precision  Recall      F1     TAR     FRR     TRR     FAR
0     0.963     0.9922  0.9697  0.9808  0.9697  0.0303  0.6667  0.3333

Natural
   Accuracy  Precision  Recall      F1     TAR     FRR  TRR  FAR
0    0.6701        1.0  0.6536  0.7905  0.6536  0.3464  1.0  0.0

Unknown
   Accuracy  Precision  Recall   F1  TAR  FRR  TRR  FAR
0       1.0        0.0     0.0  0.0  NaN  NaN  NaN  NaN


/opt/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


In [9]:
# running fixed threshold method
threshold_results = {}

for dataset_name, csv_file in datasets.items():

    result = evaluate_threshold(csv_file)

    threshold_results[dataset_name] = result

    print("\n======================")
    print(dataset_name)
    print("======================")

    print(pd.DataFrame([result]).round(4))


Degraded
   Accuracy  Precision  Recall      F1     TAR     FRR  TRR  FAR
0    0.6037        1.0  0.5947  0.7458  0.5947  0.4053  1.0  0.0

Clean
   Accuracy  Precision  Recall      F1     TAR     FRR  TRR  FAR
0    0.7333        1.0  0.7273  0.8421  0.7273  0.2727  1.0  0.0

Natural
   Accuracy  Precision  Recall      F1    TAR    FRR  TRR  FAR
0     0.119        1.0   0.075  0.1395  0.075  0.925  1.0  0.0

Unknown
   Accuracy  Precision  Recall   F1  TAR  FRR  TRR  FAR
0       1.0        0.0     0.0  0.0  NaN  NaN  NaN  NaN


/opt/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


In [10]:
# svm and single layer mlp metrics (already tested earlier)
mlp_single_results = {
    "Degraded": {
        "Accuracy": 0.938889,
        "Precision": 1.0,
        "Recall": 0.9375,
        "F1": 0.967742,
        "TAR": 0.9375,
        "FRR": 0.0625,
        "TRR": 1.0,
        "FAR": 0.0,
    },
    "Clean": {
        "Accuracy": 0.9407,
        "Precision": 1.0,
        "Recall": 0.9394,
        "F1": 0.9688,
        "TAR": 0.9394,
        "FRR": 0.0606,
        "TRR": 1.0,
        "FAR": 0.0,
    },
    "Natural": {
        "Accuracy": 0.3810,
        "Precision": 1.0,
        "Recall": 0.3500,
        "F1": 0.5185,
        "TAR": 0.3500,
        "FRR": 0.6500,
        "TRR": 1.0,
        "FAR": 0.0,
    },
    "Unknown": {"TRR": 1.0, "FAR": 0.0, "Correct Rejects": 104, "False Accepts": 0},
}

svm_results = {
    "Degraded": {
        "Accuracy": 0.974074,
        "Precision": 0.998062,
        "Recall": 0.975379,
        "F1": 0.986590,
        "TAR": 0.975379,
        "FRR": 0.024621,
        "TRR": 0.916667,
        "FAR": 0.083333,
    },
    "Clean": {
        "Accuracy": 0.970370,
        "Precision": 0.99,
        "Recall": 0.9773,
        "F1": 0.9800,
        "TAR": 0.9773,
        "FRR": 0.0227,
        "TRR": 0.6667,
        "FAR": 0.3333,
    },
    "Natural": {
        "Accuracy": 0.554422,
        "Precision": 1.0,
        "Recall": 0.532143,
        "F1": 0.694639,
        "TAR": 0.532143,
        "FRR": 0.467857,
        "TRR": 1.0,
        "FAR": 0.0,
    },
    "Unknown": {"TRR": 1.0, "FAR": 0.0, "Correct Rejects": 104, "False Accepts": 0},
}

mlp_exp1_results = {
    "Degraded": {
        "Accuracy": 0.9704,
        "Precision": 0.9981,
        "Recall": 0.9716,
        "F1": 0.9846,
        "TAR": 0.9716,
        "FRR": 0.0284,
        "TRR": 0.9167,
        "FAR": 0.0833,
    },
    "Clean": {
        "Accuracy": 0.9704,
        "Precision": 1.0000,
        "Recall": 0.9697,
        "F1": 0.9846,
        "TAR": 0.9697,
        "FRR": 0.0303,
        "TRR": 1.0000,
        "FAR": 0.0000,
    },
    "Natural": {
        "Accuracy": 0.6190,
        "Precision": 1.0000,
        "Recall": 0.6000,
        "F1": 0.7500,
        "TAR": 0.6000,
        "FRR": 0.4000,
        "TRR": 1.0000,
        "FAR": 0.0000,
    },
    "Unknown": {
        "TRR": 1.0,
        "FAR": 0.0,
        "Correct Rejects": 104,
        "False Accepts": 0,
    },
}

In [11]:
# final master comparison table
master_rows = []

for dataset_name in datasets.keys():

    for method_name, result_dict in [
        ("Threshold", threshold_results[dataset_name]),
        ("SVM", svm_results[dataset_name]),
        ("MLP Single Layer", mlp_single_results[dataset_name]),
        ("MLP Multi Layer Exp1", mlp_exp1_results[dataset_name]),
        ("MLP Multi Layer Exp2", mlp_exp2_results[dataset_name]),
    ]:

        row = {"Dataset": dataset_name, "Method": method_name}

        row.update(result_dict)

        master_rows.append(row)

master_table = pd.DataFrame(master_rows)

master_table = master_table.round(4)
display(master_table)

,Dataset,Method,Accuracy,Precision,Recall,F1,TAR,FRR,TRR,FAR,Correct Rejects,False Accepts
0,Degraded,Threshold,0.6037,1.0000,0.5947,0.7458,0.5947,0.4053,1.0000,0.0000,NaN,NaN
1,Degraded,SVM,0.9741,0.9981,0.9754,0.9866,0.9754,0.0246,0.9167,0.0833,NaN,NaN
2,Degraded,MLP Single Layer,0.9389,1.0000,0.9375,0.9677,0.9375,0.0625,1.0000,0.0000,NaN,NaN
3,Degraded,MLP Multi Layer Exp1,0.9704,0.9981,0.9716,0.9846,0.9716,0.0284,0.9167,0.0833,NaN,NaN
4,Degraded,MLP Multi Layer Exp2,0.9685,0.9981,0.9697,0.9837,0.9697,0.0303,0.9167,0.0833,NaN,NaN
5,Clean,Threshold,0.7333,1.0000,0.7273,0.8421,0.7273,0.2727,1.0000,0.0000,NaN,NaN
6,Clean,SVM,0.9704,0.9900,0.9773,0.9800,0.9773,0.0227,0.6667,0.3333,NaN,NaN
7,Clean,MLP Single Layer,0.9407,1.0000,0.9394,0.9688,0.9394,0.0606,1.0000,0.0000,NaN,NaN
8,Clean,MLP Multi Layer Exp1,0.9704,1.0000,0.9697,0.9846,0.9697,0.0303,1.0000,0.0000,NaN,NaN
9,Clean,MLP Multi Layer Exp2,0.9630,0.9922,0.9697,0.9808,0.9697,0.0303,0.6667,0.3333,NaN,NaN


In [12]:
# best model per dataset
print("\n==============================")
print("BEST MODEL PER DATASET")
print("==============================")

for dataset in [
    "Degraded",
    "Clean",
    "Natural",
]:

    temp = master_table[master_table["Dataset"] == dataset]

    best_row = temp.loc[temp["Accuracy"].idxmax()]

    print("\n---------------------")

    print("Dataset:", dataset)

    print("Winner:", best_row["Method"])

    print(
        "Accuracy:",
        round(
            best_row["Accuracy"],
            4,
        ),
    )


BEST MODEL PER DATASET

---------------------
Dataset: Degraded
Winner: SVM
Accuracy: 0.9741

---------------------
Dataset: Clean
Winner: SVM
Accuracy: 0.9704

---------------------
Dataset: Natural
Winner: MLP Multi Layer Exp2
Accuracy: 0.6701


In [13]:
# overall winner??
print("\n==============================")
print("OVERALL WINNER COUNT")
print("==============================")

winner_count = {}

for dataset in ["Degraded", "Clean", "Natural"]:

    temp = master_table[master_table["Dataset"] == dataset]

    best_row = temp.loc[temp["Accuracy"].idxmax()]

    winner = best_row["Method"]

    winner_count[winner] = winner_count.get(winner, 0) + 1

print(pd.DataFrame(winner_count.items(), columns=["Model", "Wins"]))


OVERALL WINNER COUNT
                  Model  Wins
0                   SVM     2
1  MLP Multi Layer Exp2     1
